# 53. Maximum Subarray
**Difficulty:** 🟡 Medium · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/maximum-subarray/

## 💡 Concepts

**Core concept(s):** **Kadane's algorithm** — a one-dimensional dynamic-programming sweep. Also solvable by **divide and conquer**.

**Why it applies here:** The best subarray ending at index `i` is either just `nums[i]`, or `nums[i]` appended to the best subarray ending at `i-1`. That single recurrence, tracked in a variable, yields the global answer — the essence of DP: reuse the sub-answer instead of recomputing it.

**Key intuition / mental model:** Carry a running sum. If it ever goes negative it can only hurt what comes next, so drop it and restart from the current element.

---

### 📚 What is Dynamic Programming (DP)?
**DP** solves a problem by combining answers to overlapping subproblems and *reusing* them instead of recomputing. Kadane is DP where the only subproblem needed is "best sum ending here", so O(1) memory suffices.
- **Complexity here:** O(n) time, O(1) space.

### 📚 What is Divide and Conquer?
**Divide and conquer** splits the input in half, solves each half recursively, and merges. For max-subarray the merge also checks subarrays that *cross* the midpoint.
- **Complexity:** T(n) = 2T(n/2) + O(n) → **O(n log n)**.

## 📝 Problem

Find the contiguous subarray with the largest sum and return that sum. The array has at least one element (may be all negative).

**Example**
```
Input:  nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
Output: 6        # subarray [4, -1, 2, 1]
```
**Constraints:** `1 <= len(nums) <= 10^5`.

### Approach 1 — Brute Force (worst)

**Idea:** For every start `i`, extend a running sum over `j >= i` and track the max. (The naive O(n³) triple loop is avoided by summing incrementally.)

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def max_subarray_brute(nums: List[int]) -> int:
    best = float("-inf")
    n = len(nums)
    for i in range(n):                     # start of the subarray
        run = 0
        for j in range(i, n):              # extend the subarray one element at a time
            run += nums[j]                 # running sum of nums[i..j]
            best = max(best, run)          # track the largest sum seen
    return best

### Approach 2 — Divide and Conquer (better)

**Idea:** Split in half. The answer is the best of: max subarray fully in the left, fully in the right, or one that **crosses** the middle (best suffix of left + best prefix of right).

**Time complexity:** `O(n log n)`.

**Space complexity:** `O(log n)` recursion stack.

In [ ]:
from typing import List

def max_subarray_divide(nums: List[int]) -> int:
    def solve(lo, hi):
        if lo == hi:
            return nums[lo]                # a single element is its own best subarray
        mid = (lo + hi) // 2
        left = solve(lo, mid)              # best subarray fully in the left half
        right = solve(mid + 1, hi)         # best subarray fully in the right half
        # Best subarray that CROSSES the middle = best suffix of left + best prefix of right.
        s, best_left = 0, float("-inf")
        for i in range(mid, lo - 1, -1):   # extend leftward from the middle
            s += nums[i]; best_left = max(best_left, s)
        s, best_right = 0, float("-inf")
        for i in range(mid + 1, hi + 1):   # extend rightward from the middle
            s += nums[i]; best_right = max(best_right, s)
        return max(left, right, best_left + best_right)
    return solve(0, len(nums) - 1)

### Approach 3 — Kadane's Algorithm (optimal)

**Idea:** Track `cur` = best sum of a subarray ending at the current index. Extend it, or restart at the current element if extending would drop below it. Keep the global max.

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def max_subarray_kadane(nums: List[int]) -> int:
    cur = best = nums[0]                    # cur = best sum of a subarray ENDING at this index
    for x in nums[1:]:
        cur = max(x, cur + x)              # extend the previous run, or start fresh at x
        best = max(best, cur)              # track the best run seen anywhere
    return best

In [ ]:
# Correctness check
tests = [
    ([-2, 1, -3, 4, -1, 2, 1, -5, 4], 6),
    ([1], 1),
    ([-3, -1, -2], -1),                    # all negative -> the least-bad single element
    ([5, 4, -1, 7, 8], 23),
]
for nums, expected in tests:
    b = max_subarray_brute(nums)
    d = max_subarray_divide(nums)
    k = max_subarray_kadane(nums)
    print(f"{nums} -> brute={b}, divide={d}, kadane={k} | expected={expected}")
    assert b == d == k == expected, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    # alternating signs -> no run dominates, every approach does full work
    nums = [(-1) ** i * (i % 7 + 1) for i in range(n)]
    return (nums,)

solutions = {
    "brute  O(n^2)    ": max_subarray_brute,
    "divide O(n log n)": max_subarray_divide,
    "kadane O(n)      ": max_subarray_kadane,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Kadane / 1-D DP:** When an optimum over contiguous ranges depends only on the best answer ending at the previous index, carry it in one variable — O(n), O(1).
- **"Drop the negative prefix":** A running total that turns negative can only reduce future sums; reset it.
- **Signal to reach for it:** "maximum/minimum sum of a contiguous subarray", "best streak", "max profit with reset".
- **Related problems:** Maximum Product Subarray, Best Time to Buy/Sell Stock, Maximum Circular Subarray, House Robber.
- **Common pitfalls:** (1) initializing `best = 0`, which is wrong for all-negative arrays — seed with `nums[0]`; (2) confusing "subarray" (contiguous) with "subsequence".